In [ ]:
# 04_digital_twin.ipynb

# ---- Cell 1: Install and imports ----
!pip install -q ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---- Cell 2: Load outputs from notebooks 02 and 03 ----
# Upload these files to the Colab session (left panel > folder icon > upload),
# or mount Google Drive and point the paths there.
model_data = pd.read_csv('ujani_model_dataset.csv', parse_dates=['date'])
forecast = pd.read_csv('ujani_forecast_6month.csv', parse_dates=['date'])

print(f'Historical months: {len(model_data)}')
print(f'Forecast months:   {len(forecast)}')
print(model_data[['date', 'water_km2', 'rain_mm']].tail())

# ---- Cell 3: Reservoir state model (assumptions go here) ----
# These are ASSUMPTIONS, not measured values. Replace with published figures for Ujani
# when available (Maharashtra Water Resources Department or CWC reports).
RESERVOIR = {
    'name': 'Ujani (approximate model)',
    'mean_depth_m': 10.0,         # assumed average depth across the water surface
    'full_capacity_mcm': 117.0,   # assumed full storage in million cubic metres (MCM)
    'catchment_km2': 1200.0,      # assumed catchment area draining to the reservoir
    'runoff_coefficient': 0.15,   # share of rainfall that becomes inflow (rough estimate)
    'evap_mm_per_month': 150.0,   # assumed open-water evaporation
}

def area_to_volume_mcm(area_km2, depth_m=RESERVOIR['mean_depth_m']):
    """Approximate stored volume from water-surface area and an average depth."""
    return area_km2 * depth_m / 1000.0   # km² × m = million m³ (MCM)

def rain_to_inflow_mcm(rain_mm, catchment_km2=RESERVOIR['catchment_km2'],
                       coeff=RESERVOIR['runoff_coefficient']):
    """Monthly catchment rainfall (mm) to inflow (MCM)."""
    return rain_mm / 1000.0 * catchment_km2 * 1e6 / 1e6 * coeff   # mm -> m, km² -> m²

# ---- Cell 4: Starting state from the latest observation ----
latest = model_data.iloc[-1]
start_volume = area_to_volume_mcm(latest['water_km2'])
print(f"Starting volume (estimate): {start_volume:.1f} MCM "
      f"({start_volume / RESERVOIR['full_capacity_mcm'] * 100:.0f}% of assumed capacity)")

# ---- Cell 5: Simulation function (water balance, month by month) ----
def simulate(months, rain_scale=1.0, release_scale=1.0,
             base_release_mcm=None, start=start_volume):
    """
    Water balance: next volume = current + inflow - release - evaporation.
    rain_scale:    multiplies rainfall (0.8 = 20% drier than the forecast assumes)
    release_scale: multiplies the release (1.15 = 15% more water released)
    """
    # Default release: half of the average monthly inflow, a placeholder policy
    if base_release_mcm is None:
        base_release_mcm = model_data['rain_mm'].mean() / 1000 * RESERVOIR['catchment_km2'] \
                           * RESERVOIR['runoff_coefficient'] * 0.5

    vol = start
    rows = []
    for _, r in months.iterrows():
        rain = r['rain_mm'] * rain_scale
        inflow = rain_to_inflow_mcm(rain)
        evap = RESERVOIR['evap_mm_per_month'] / 1000 * (vol / RESERVOIR['mean_depth_m']) * 1e3 / 1e3
        release = base_release_mcm * release_scale
        vol = vol + inflow - release - evap
        vol = float(np.clip(vol, 0, RESERVOIR['full_capacity_mcm']))   # cannot go below zero or above capacity
        rows.append({'date': r['date'], 'rain_mm': rain, 'inflow_mcm': inflow,
                     'release_mcm': release, 'evap_mcm': evap, 'volume_mcm': vol,
                     'percent_full': vol / RESERVOIR['full_capacity_mcm'] * 100})
    return pd.DataFrame(rows)

# Use the 6-month forecast horizon. Its rainfall is the historical average (from notebook 03),
# so scenarios test changes against that baseline.
horizon = forecast.rename(columns={'water_km2_forecast': 'water_km2'}).copy()
horizon = horizon.merge(model_data[['date', 'rain_mm']], how='left', on='date')
horizon['rain_mm'] = horizon['rain_mm'].fillna(model_data.groupby(model_data['date'].dt.month)['rain_mm'].mean()[horizon['date'].dt.month].values)

baseline = simulate(horizon)
print(baseline[['date', 'volume_mcm', 'percent_full']].round(1).to_string(index=False))

# ---- Cell 6: Interactive scenario panel with sliders ----
rain_slider = widgets.FloatSlider(value=1.0, min=0.5, max=1.5, step=0.05,
                                  description='Rainfall x', readout_format='.2f')
release_slider = widgets.FloatSlider(value=1.0, min=0.5, max=1.5, step=0.05,
                                     description='Release x', readout_format='.2f')
output = widgets.Output()

def update(_=None):
    scen = simulate(horizon, rain_scale=rain_slider.value, release_scale=release_slider.value)
    with output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 4.5))
        ax.plot(baseline['date'], baseline['percent_full'], 'o--', label='Baseline (average rain, base release)')
        ax.plot(scen['date'], scen['percent_full'], 'o-', label='Scenario')
        ax.axhline(20, color='red', linestyle=':', label='Warning level (20%, illustrative)')
        ax.set_ylabel('% of assumed capacity')
        ax.set_title('Ujani water-balance scenario')
        ax.set_ylim(0, 100)
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()
        end_base = baseline['percent_full'].iloc[-1]
        end_scen = scen['percent_full'].iloc[-1]
        print(f'End-of-horizon storage: baseline {end_base:.1f}%  |  scenario {end_scen:.1f}%  '
              f'(change {end_scen - end_base:+.1f} points)')

rain_slider.observe(update, names='value')
release_slider.observe(update, names='value')

display(widgets.VBox([rain_slider, release_slider, output]))
update()

# ---- Cell 7: Save the twin's outputs for the viewer ----
baseline.to_csv('ujani_twin_baseline.csv', index=False)
print('Saved ujani_twin_baseline.csv')